In [ ]:
# SELF-CONTAINED - Works on Codespace AND Shadow PC
# No external imports - everything embedded

import yfinance as yf
import pandas as pd
import numpy as np
import time
import json
import sqlite3
import os
from datetime import datetime, timedelta
from pathlib import Path
from IPython.display import display, HTML, clear_output

# Auto-detect environment (Codespace vs Shadow PC)
if os.path.exists('/workspaces/quantum-ai-trader_v1.1'):
    WORKSPACE = '/workspaces/quantum-ai-trader_v1.1'
    ENV = 'Codespace'
elif os.path.exists(r'C:\Users\Shadow\quantum-ai-trader_v1.1'):
    WORKSPACE = r'C:\Users\Shadow\quantum-ai-trader_v1.1'
    ENV = 'Shadow PC'
else:
    # Fallback to notebook's parent directory
    WORKSPACE = str(Path.cwd().parent)
    ENV = 'Unknown'

DATA_DIR = os.path.join(WORKSPACE, 'data')
DB_PATH = os.path.join(DATA_DIR, 'trading_system.db')
CHECKPOINT_FILE = os.path.join(DATA_DIR, 'ohlcv_checkpoint.json')

# Ensure directories exist
os.makedirs(DATA_DIR, exist_ok=True)

print(f"✅ Environment: {ENV}")
print(f"📁 Workspace: {WORKSPACE}")
print(f"💾 Database: {DB_PATH}")
print(f"🕐 Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# DATABASE CLASS - EMBEDDED (no external imports needed)

class TradingDatabase:
    """Production database - embedded in notebook for portability."""
    
    def __init__(self, db_path=DB_PATH):
        self.db_path = db_path
        self.conn = None
        
    def connect(self):
        """Connect to database with optimizations."""
        self.conn = sqlite3.connect(self.db_path, timeout=30.0)
        self.conn.execute("PRAGMA journal_mode=WAL")
        self.conn.execute("PRAGMA synchronous=NORMAL")
        self.conn.execute("PRAGMA cache_size=-64000")
        return self.conn
    
    def close(self):
        if self.conn:
            self.conn.close()
            self.conn = None
    
    def create_schema(self):
        """Create database tables."""
        cursor = self.conn.cursor()
        
        # Tickers table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS tickers (
                ticker TEXT PRIMARY KEY,
                sector TEXT,
                industry TEXT,
                market_cap_category TEXT,
                notes TEXT,
                active INTEGER DEFAULT 1,
                added_date TEXT DEFAULT CURRENT_TIMESTAMP
            )
        """)
        
        # OHLCV table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS ohlcv_daily (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                ticker TEXT NOT NULL,
                date TEXT NOT NULL,
                open REAL, high REAL, low REAL, close REAL,
                volume INTEGER, adj_close REAL,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP,
                UNIQUE(ticker, date)
            )
        """)
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_ohlcv_ticker_date ON ohlcv_daily(ticker, date)")
        
        self.conn.commit()
    
    def load_ticker_universe(self, csv_path):
        """Load tickers from CSV - handles any CSV format."""
        if not os.path.exists(csv_path):
            print(f"⚠️ CSV not found at {csv_path}")
            return 0
        
        df = pd.read_csv(csv_path)
        
        # Ensure 'ticker' column exists
        if 'ticker' not in df.columns:
            # Try common alternatives
            if 'symbol' in df.columns:
                df = df.rename(columns={'symbol': 'ticker'})
            elif 'Ticker' in df.columns:
                df = df.rename(columns={'Ticker': 'ticker'})
            else:
                print(f"❌ No ticker column found in CSV")
                return 0
        
        # Add missing columns with defaults
        if 'sector' not in df.columns:
            df['sector'] = 'Unknown'
        if 'industry' not in df.columns:
            df['industry'] = 'Unknown'
        if 'market_cap_category' not in df.columns:
            df['market_cap_category'] = 'Unknown'
        if 'notes' not in df.columns:
            df['notes'] = ''
        if 'active' not in df.columns:
            df['active'] = 1
        if 'added_date' not in df.columns:
            df['added_date'] = datetime.now().isoformat()
        
        # Keep only needed columns
        cols = ['ticker', 'sector', 'industry', 'market_cap_category', 'notes', 'active', 'added_date']
        df = df[cols]
        
        # Insert into database
        df.to_sql('tickers', self.conn, if_exists='replace', index=False)
        self.conn.commit()
        
        return len(df)
    
    def get_active_tickers(self):
        """Get all active tickers."""
        query = "SELECT ticker FROM tickers WHERE active = 1 ORDER BY ticker"
        return pd.read_sql_query(query, self.conn)['ticker'].tolist()
    
    def insert_ohlcv_batch(self, ticker, ohlcv_df):
        """Insert OHLCV data."""
        if ohlcv_df.empty:
            return 0
        
        ohlcv_df = ohlcv_df.reset_index()
        ohlcv_df['ticker'] = ticker
        ohlcv_df['date'] = pd.to_datetime(ohlcv_df['Date']).dt.strftime('%Y-%m-%d')
        ohlcv_df['created_at'] = datetime.now().isoformat()
        
        col_mapping = {
            'Open': 'open', 'High': 'high', 'Low': 'low',
            'Close': 'close', 'Volume': 'volume', 'Adj Close': 'adj_close'
        }
        ohlcv_df = ohlcv_df.rename(columns=col_mapping)
        
        cols = ['ticker', 'date', 'open', 'high', 'low', 'close', 'volume', 'adj_close', 'created_at']
        insert_df = ohlcv_df[cols]
        
        # Use executemany for reliable inserts (avoids column name errors)
        cursor = self.conn.cursor()
        cursor.executemany(
            "INSERT OR IGNORE INTO ohlcv_daily (ticker, date, open, high, low, close, volume, adj_close, created_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
            insert_df.values.tolist()
        )
        self.conn.commit()
        return len(insert_df)
    
    def get_latest_ohlcv_date(self, ticker):
        """Get most recent date for ticker."""
        query = "SELECT MAX(date) as max_date FROM ohlcv_daily WHERE ticker = ?"
        result = pd.read_sql_query(query, self.conn, params=[ticker])
        return result['max_date'].iloc[0] if not result.empty else None
    
    def get_database_stats(self):
        """Get database statistics."""
        cursor = self.conn.cursor()
        stats = {}
        
        cursor.execute("SELECT COUNT(*) FROM tickers WHERE active = 1")
        stats['active_tickers'] = cursor.fetchone()[0]
        
        cursor.execute("""
            SELECT COUNT(DISTINCT ticker) as tickers_with_data,
                   MIN(date) as earliest_date,
                   MAX(date) as latest_date,
                   COUNT(*) as total_bars
            FROM ohlcv_daily
        """)
        row = cursor.fetchone()
        stats['ohlcv_tickers'] = row[0]
        stats['ohlcv_earliest'] = row[1]
        stats['ohlcv_latest'] = row[2]
        stats['ohlcv_total_bars'] = row[3]
        
        return stats

# Initialize database
db = TradingDatabase()
db.connect()
db.create_schema()

# Load ticker universe
UNIVERSE_FILE = os.path.join(DATA_DIR, 'ticker_universe_300.csv')
ticker_count = db.load_ticker_universe(UNIVERSE_FILE)

# Get all tickers
all_tickers = db.get_active_tickers()
print(f"\n📊 Total universe: {len(all_tickers)} tickers")
print(f"Sample: {all_tickers[:20]}")

# Check current database state
stats = db.get_database_stats()
print(f"\n📁 Current database state:")
print(f"   Tickers with data: {stats['ohlcv_tickers']}")
print(f"   Total bars: {stats['ohlcv_total_bars']:,}")

if stats['ohlcv_tickers'] > 0:
    print(f"   Date range: {stats['ohlcv_earliest']} to {stats['ohlcv_latest']}")


In [ ]:
# Checkpoint management
CHECKPOINT_FILE = os.path.join(DATA_DIR, 'ohlcv_checkpoint.json')

def load_checkpoint():
    """Load previous progress."""
    try:
        with open(CHECKPOINT_FILE, 'r') as f:
            checkpoint = json.load(f)
            return set(checkpoint['processed']), checkpoint.get('failed', [])
    except FileNotFoundError:
        return set(), []
    except Exception as e:
        print(f"⚠️ Checkpoint load error: {e}")
        return set(), []

def save_checkpoint(processed_set, failed_list, success_count, total_count):
    """Save current progress."""
    checkpoint = {
        'timestamp': datetime.now().isoformat(),
        'processed': list(processed_set),
        'failed': failed_list,
        'success_count': success_count,
        'total_processed': total_count
    }
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint, f, indent=2)

# Load existing checkpoint
processed_set, failed_list = load_checkpoint()
print(f"📁 Checkpoint loaded: {len(processed_set)} already processed")

remaining = [t for t in all_tickers if t not in processed_set]
print(f"🔄 Remaining to process: {len(remaining)} tickers")

## 🚀 START DATA COLLECTION

**This cell will run for 3-4 hours.**

- Progress updates every 10 tickers
- Checkpoint saved every 50 tickers
- Safe to interrupt - will resume from checkpoint

**You can walk away. It will finish.**

In [ ]:
# MAIN DATA COLLECTION LOOP
# This runs for 3-4 hours - safe to walk away

print("="*60)
print("🚀 STARTING DATA COLLECTION")
print("="*60)
print(f"Total tickers: {len(all_tickers)}")
print(f"Remaining: {len(remaining)}")
print(f"Already done: {len(processed_set)}")
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

start_time = time.time()
success_count = len(processed_set)
total_processed = len(processed_set)

for i, ticker in enumerate(remaining, 1):
    total_processed += 1
    
    try:
        # Check if we already have recent data
        latest_date = db.get_latest_ohlcv_date(ticker)
        
        if latest_date:
            latest_dt = datetime.strptime(latest_date, '%Y-%m-%d')
            days_ago = (datetime.now() - latest_dt).days
            
            if days_ago < 7:
                print(f"[{i}/{len(remaining)}] {ticker} - up-to-date (last: {latest_date})")
                processed_set.add(ticker)
                success_count += 1
                time.sleep(0.2)
                continue
        
        # Download from yfinance
        data = yf.download(ticker, period='2y', progress=False, auto_adjust=False)
        
        if data.empty or len(data) < 20:
            print(f"[{i}/{len(remaining)}] {ticker} - ❌ No data ({len(data)} bars)")
            failed_list.append(ticker)
            continue
        
        # Insert into database
        rows = db.insert_ohlcv_batch(ticker, data)
        print(f"[{i}/{len(remaining)}] {ticker} - ✅ {rows} bars")
        
        processed_set.add(ticker)
        success_count += 1
        
    except Exception as e:
        print(f"[{i}/{len(remaining)}] {ticker} - ❌ Error: {str(e)[:50]}")
        failed_list.append(ticker)
    
    # Progress update every 10 tickers
    if i % 10 == 0:
        elapsed = time.time() - start_time
        rate = i / elapsed if elapsed > 0 else 0
        remaining_count = len(remaining) - i
        eta = remaining_count / rate if rate > 0 else 0
        
        print("\n" + "="*60)
        print(f"📊 PROGRESS: {len(processed_set)}/{len(all_tickers)} ({len(processed_set)/len(all_tickers)*100:.1f}%)")
        print(f"✅ Success: {success_count} | ❌ Failed: {len(failed_list)}")
        print(f"⏱️  Rate: {rate*60:.1f} tickers/min")
        print(f"🕐 ETA: {eta/60:.1f} min ({eta/3600:.2f} hours)")
        print(f"🕐 Current time: {datetime.now().strftime('%H:%M:%S')}")
        print("="*60 + "\n")
    
    # Save checkpoint every 50 tickers
    if i % 50 == 0:
        save_checkpoint(processed_set, failed_list, success_count, total_processed)
        print(f"\n📁 CHECKPOINT SAVED at {len(processed_set)} tickers\n")
    
    # Polite delay
    time.sleep(0.5)

# Final checkpoint
save_checkpoint(processed_set, failed_list, success_count, total_processed)

# Summary
elapsed_total = time.time() - start_time
print("\n" + "="*60)
print("✅ DATA COLLECTION COMPLETE")
print("="*60)
print(f"Total processed: {total_processed}")
print(f"Successful: {success_count}")
print(f"Failed: {len(failed_list)}")
print(f"Success rate: {success_count/total_processed*100:.1f}%")
print(f"Total runtime: {elapsed_total/60:.1f} min ({elapsed_total/3600:.2f} hours)")
print(f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

In [ ]:
# VALIDATION CHECK - Did we get good data?

stats = db.get_database_stats()

print("📊 FINAL DATABASE STATE")
print("="*60)
print(f"Active tickers:          {stats['active_tickers']}")
print(f"Tickers with data:       {stats['ohlcv_tickers']}")
print(f"Total OHLCV bars:        {stats['ohlcv_total_bars']:,}")
print(f"Date range:              {stats['ohlcv_earliest']} to {stats['ohlcv_latest']}")
print("="*60)

# KILL SWITCH: Check data quality
coverage = stats['ohlcv_tickers'] / stats['active_tickers'] if stats['active_tickers'] > 0 else 0

print(f"\n🔍 VALIDATION:")
print(f"   Coverage: {coverage*100:.1f}%")
print(f"   Failed tickers (bankrupt/delisted): {len(failed_list)}")

if coverage < 0.85:
    print(f"\n❌ KILL SWITCH: Coverage {coverage*100:.1f}% < 85%")
    print(f"   Data quality insufficient. DO NOT PROCEED.")
    print(f"   Review failed tickers: {failed_list[:20]}")
else:
    print(f"\n✅ VALIDATION PASSED: {coverage*100:.1f}% coverage ({stats['ohlcv_tickers']} tickers)")
    print(f"   {stats['ohlcv_total_bars']:,} total bars ready for analysis")
    print(f"\n🎯 NEXT STEP: Run DAY2_EVENT_LAG_SCANNER.ipynb")
    print(f"   That's where we find the BEST 10-20 patterns from these {stats['ohlcv_tickers']} tickers")

db.close()

## 📈 QUICK ANALYSIS - What Did We Find?

Let's peek at the data and see initial patterns.

In [ ]:
# Reconnect to analyze
db = TradingDatabase()
db.connect()

# Get sample of tickers with most data
query = """
    SELECT ticker, COUNT(*) as bar_count, 
           MIN(date) as earliest, MAX(date) as latest
    FROM ohlcv_daily
    GROUP BY ticker
    ORDER BY bar_count DESC
    LIMIT 20
"""

top_coverage = pd.read_sql_query(query, db.conn)
print("📊 Top 20 tickers by data coverage:")
print(top_coverage)

db.close()

---
## ✅ DATA COLLECTION COMPLETE

**Next steps:**
1. Build volume anomaly scanner (finds unusual activity)
2. Build SEC filing monitor (catches redemptions/mergers)
3. Analyze data to find:
   - 💀 Dying names (downtrend, no volume)
   - 📈 Steady gainers (consistent uptrend)
   - 🚀 Emerging names (breakout patterns)

**Your father's legacy continues.**